# installation

In [1]:
import os

!pip install -q pyspark==3.4.1 #delta-spark==2.4.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 MB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 9.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.4.1 which is incompatible.


In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.getOrCreate()
spark

In [75]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import Window

# loading dataframe

In [90]:
customer=spark.read.format("csv")\
        .option("inferSchema","true").option("header","true")\
        .option("sep",",").load("/content/dim_customer.csv")

product=spark.read.format("csv")\
        .option("inferSchema","true").option("header","true")\
        .option("sep",",").load("/content/dim_product.csv")

store=spark.read.format("csv")\
        .option("inferSchema","true").option("header","true")\
        .option("sep",",").load("/content/dim_store.csv")

sales=spark.read.format("csv")\
        .option("inferSchema","true").option("header","true")\
        .option("sep",",").load("/content/fact_sales.csv")

In [19]:
customer.show(3)

+-----------+-------------+------+-----+------+----------+
|customer_id|customer_name|  city|state|gender| join_date|
+-----------+-------------+------+-----+------+----------+
|          1|         Amit|Mumbai|   MH|     M|2023-01-10|
|          2|        Priya|  Pune|   MH|     F|2023-02-15|
|          3|        Rahul| Delhi|   DL|     M|2023-03-20|
+-----------+-------------+------+-----+------+----------+
only showing top 3 rows



In [92]:

product.show(3)

+----------+------------+--------+-------+
|product_id|product_name|category|  brand|
+----------+------------+--------+-------+
|       101|      iPhone|  Mobile|  Apple|
|       102|      Galaxy|  Mobile|Samsung|
|       103|     MacBook|  Laptop|  Apple|
+----------+------------+--------+-------+
only showing top 3 rows



In [25]:
store.show(3)

+--------+--------------+------+------+
|store_id|    store_name|  city|region|
+--------+--------------+------+------+
|       1|Mumbai Central|Mumbai|  West|
|       2|     Pune Mall|  Pune|  West|
|       3|     Delhi Hub| Delhi| North|
+--------+--------------+------+------+
only showing top 3 rows



In [23]:
sales.show(3)

+-------+-----------+----------+--------+----------+--------+----------+
|sale_id|customer_id|product_id|store_id| sale_date|quantity|unit_price|
+-------+-----------+----------+--------+----------+--------+----------+
|      1|          1|       101|       1|2024-01-01|       1|     80000|
|      2|          2|       102|       2|2024-01-02|       1|     70000|
|      3|          3|       103|       3|2024-01-03|       1|    120000|
+-------+-----------+----------+--------+----------+--------+----------+
only showing top 3 rows



# Problem

## Begineer

### 1. Create `sale_amount = quantity * unit_price`.

In [29]:
sales.withColumn("sale_amount",col("quantity")*col("unit_price")).show(3)

+-------+-----------+----------+--------+----------+--------+----------+-----------+
|sale_id|customer_id|product_id|store_id| sale_date|quantity|unit_price|sale_amount|
+-------+-----------+----------+--------+----------+--------+----------+-----------+
|      1|          1|       101|       1|2024-01-01|       1|     80000|      80000|
|      2|          2|       102|       2|2024-01-02|       1|     70000|      70000|
|      3|          3|       103|       3|2024-01-03|       1|    120000|     120000|
+-------+-----------+----------+--------+----------+--------+----------+-----------+
only showing top 3 rows



### 2. Find total revenue.

In [34]:
sales.select(sum(col("unit_price")*col("quantity")).alias("total_revenue")).show()

+-------------+
|total_revenue|
+-------------+
|      4385000|
+-------------+



### 3. Find average sale amount.

In [36]:
sales.select(
    avg(col("unit_price")*col("quantity"))
    ).show()

+----------------------------+
|avg((unit_price * quantity))|
+----------------------------+
|           66439.39393939394|
+----------------------------+



### 4. Find total revenue by customer.

In [41]:
sale.groupBy(col("customer_id"))\
    .agg(
        sum(col("unit_price")*col("quantity")).alias("total_revenue")
    ).show(3)

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|         12|       260000|
|          1|       650000|
|         13|       210000|
+-----------+-------------+
only showing top 3 rows



### 5. Find total revenue by product.

In [43]:
sales.groupBy(col("product_id"))\
      .agg(
          sum(col("unit_price")*col("quantity")).alias("total_revenue")
      ).show(3)

+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|       108|       350000|
|       101|       560000|
|       103|       960000|
+----------+-------------+
only showing top 3 rows



### 6. Find total revenue by store.

In [45]:
sales.groupBy(col("store_id"))\
      .agg(
          sum(col("quantity")*col("unit_price")).alias("total_revenue")
          ).show(3)

+--------+-------------+
|store_id|total_revenue|
+--------+-------------+
|       1|      1045000|
|       3|       800000|
|       5|       900000|
+--------+-------------+
only showing top 3 rows



### 7. Count customers by city.

In [52]:
customer.groupBy(col("city")).agg(count("*").alias("count")).show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    3|
|  Chennai|    3|
|   Mumbai|    5|
|     Pune|    4|
|    Delhi|    5|
|Hyderabad|    3|
+---------+-----+



### 8. Count products by category.

In [54]:
product.groupBy(col("category")).agg(count("*").alias("count")).show()


+-----------+-----+
|   category|count|
+-----------+-----+
|Electronics|    5|
|  Accessory|    2|
|     Laptop|    2|
|     Mobile|    2|
|     Tablet|    2|
+-----------+-----+



### 9. Find total quantity sold by product.

In [57]:
sales.groupBy(col("product_id")).agg(sum(col("quantity")).alias("total_quantity")).show(3)

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       108|             7|
|       101|             7|
|       103|             8|
+----------+--------------+
only showing top 3 rows



### 10. Find sales made in May 2024.

In [67]:
sales.filter(
    (month(col("sale_date"))==5 ) & (year(col("sale_date"))==2024)
).show(3)

+-------+-----------+----------+--------+----------+--------+----------+
|sale_id|customer_id|product_id|store_id| sale_date|quantity|unit_price|
+-------+-----------+----------+--------+----------+--------+----------+
|     41|          1|       103|       1|2024-05-01|       1|    120000|
|     42|          2|       104|       2|2024-05-02|       1|     90000|
|     43|          3|       105|       3|2024-05-03|       2|     15000|
+-------+-----------+----------+--------+----------+--------+----------+
only showing top 3 rows



### 11. Find customers joined in 2024.

In [69]:
customer.filter(year(col("join_date"))==2024).show(3)

+-----------+-------------+---------+-----+------+----------+
|customer_id|customer_name|     city|state|gender| join_date|
+-----------+-------------+---------+-----+------+----------+
|         13|        Varun|Hyderabad|   TS|     M|2024-01-07|
|         14|        Pooja|    Delhi|   DL|     F|2024-01-20|
|         15|       Nikhil|     Pune|   MH|     M|2024-02-11|
+-----------+-------------+---------+-----+------+----------+
only showing top 3 rows



### 12. Find products belonging to Electronics category.

In [72]:
product.filter(col("category")=="Electronics").show(3)

+----------+---------------+-----------+-----+
|product_id|   product_name|   category|brand|
+----------+---------------+-----------+-----+
|       107|             TV|Electronics| Sony|
|       108|   Refrigerator|Electronics|   LG|
|       109|Washing Machine|Electronics|   LG|
+----------+---------------+-----------+-----+
only showing top 3 rows



### 13. Find top 5 products by quantity sold.

In [85]:
sales.groupBy(col("product_id"))\
      .agg(sum(col("quantity")).alias("total_quantity"))\
      .orderBy(col("total_quantity").desc()).show(5)

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|       105|            17|
|       110|             9|
|       103|             8|
|       102|             8|
|       109|             8|
+----------+--------------+
only showing top 5 rows



### 14. Find top 5 customers by revenue.

In [86]:
sales.groupBy(col("customer_id")).agg(
    sum(col("quantity")*col("unit_price")).alias("total_revenue")
).orderBy(col("total_revenue").desc()).show(5)

+-----------+-------------+
|customer_id|total_revenue|
+-----------+-------------+
|          1|       650000|
|          2|       575000|
|          3|       420000|
|          5|       395000|
|         12|       260000|
+-----------+-------------+
only showing top 5 rows



### 15. Find category-wise revenue.

In [97]:
sales.alias("s").join(product.alias("p"),
                     col("s.product_id")==col("p.product_id"),
                      "inner")\
                .groupBy(col("p.category")).agg(sum(col("quantity")*col("unit_price")).alias("total_revenue"))\
                .show()


+-----------+-------------+
|   category|total_revenue|
+-----------+-------------+
|Electronics|      1310000|
|  Accessory|       455000|
|     Laptop|      1500000|
|     Mobile|      1120000|
+-----------+-------------+



## Intermediate

### 16. Show customer name with total revenue.

In [102]:
sales.alias("s").join(customer.alias("c"),
                      col("s.customer_id")==col("c.customer_id"),
                      "inner"
              ).groupBy(col("s.customer_id"),col("c.customer_name"))\
              .agg(sum(col("s.quantity")*col("s.unit_price")).alias("total_revenue")).show(3)

+-----------+-------------+-------------+
|customer_id|customer_name|total_revenue|
+-----------+-------------+-------------+
|          5|        Vikas|       395000|
|         19|       Manish|        65000|
|         10|       Anjali|       190000|
+-----------+-------------+-------------+
only showing top 3 rows



### 17. Find customers with no sales.

In [103]:
customer.alias("c")\
        .join(
            sales.alias("s"),
            col("c.customer_id")==col("s.customer_id"),
            "left"
        )\
        .filter(col("s.customer_id").isNull()).show()

+-----------+-------------+------+-----+------+----------+-------+-----------+----------+--------+---------+--------+----------+
|customer_id|customer_name|  city|state|gender| join_date|sale_id|customer_id|product_id|store_id|sale_date|quantity|unit_price|
+-----------+-------------+------+-----+------+----------+-------+-----------+----------+--------+---------+--------+----------+
|         21|          Dev|Mumbai|   MH|     M|2024-05-01|   null|       null|      null|    null|     null|    null|      null|
|         22|       Ananya|  Pune|   MH|     F|2024-05-05|   null|       null|      null|    null|     null|    null|      null|
|         23|       Sachin| Delhi|   DL|     M|2024-05-10|   null|       null|      null|    null|     null|    null|      null|
+-----------+-------------+------+-----+------+----------+-------+-----------+----------+--------+---------+--------+----------+



In [105]:
customer.alias("c")\
        .join(
            sales.alias("s"),
            col("c.customer_id")==col("s.customer_id"),
            "left_anti"
        ).show()

+-----------+-------------+------+-----+------+----------+
|customer_id|customer_name|  city|state|gender| join_date|
+-----------+-------------+------+-----+------+----------+
|         21|          Dev|Mumbai|   MH|     M|2024-05-01|
|         22|       Ananya|  Pune|   MH|     F|2024-05-05|
|         23|       Sachin| Delhi|   DL|     M|2024-05-10|
+-----------+-------------+------+-----+------+----------+



### 18. Find products with no sales.

In [106]:
product.alias("p")\
        .join(
            sales.alias("s"),
            col("p.product_id")==col("s.product_id"),
            "left_anti"
        ).show()

+----------+------------+-----------+---------+
|product_id|product_name|   category|    brand|
+----------+------------+-----------+---------+
|       111|        iPad|     Tablet|    Apple|
|       112| Surface Pro|     Tablet|Microsoft|
|       113|     Printer|Electronics|       HP|
+----------+------------+-----------+---------+



### 19. Find stores with no sales.

In [111]:
store.alias("st")\
      .join(
          sales.alias("s"),
          col("st.store_id")==col("s.store_id"),
          "left"
      ).filter(col("s.store_id").isNull()).show(3)

+--------+----------+----+------+-------+-----------+----------+--------+---------+--------+----------+
|store_id|store_name|city|region|sale_id|customer_id|product_id|store_id|sale_date|quantity|unit_price|
+--------+----------+----+------+-------+-----------+----------+--------+---------+--------+----------+
+--------+----------+----+------+-------+-----------+----------+--------+---------+--------+----------+



In [112]:
store.alias("st")\
      .join(
          sales.alias("s"),
          col("st.store_id")==col("s.store_id"),
          "left_anti"
      ).show(3)

+--------+----------+----+------+
|store_id|store_name|city|region|
+--------+----------+----+------+
+--------+----------+----+------+



### 20. Find duplicate customer records.

In [113]:
customer.groupBy(col("customer_id")).agg(count("*").alias("cnt")).filter(col("cnt")>1).show()

+-----------+---+
|customer_id|cnt|
+-----------+---+
+-----------+---+



### 21. Find duplicate sales records.

In [116]:
sales.groupBy(col("sale_id")).agg(count("*").alias("cnt")).filter(col("cnt")>1).show()

+-------+---+
|sale_id|cnt|
+-------+---+
+-------+---+



### 22. Find customers who purchased more than one product.

In [118]:
sales.groupBy(col("customer_id"))\
    .agg(countDistinct(col("product_id")).alias("cnt"))\
    .filter(col("cnt")>1).show(2)


+-----------+---+
|customer_id|cnt|
+-----------+---+
|         12|  2|
|          1|  7|
+-----------+---+
only showing top 2 rows



In [119]:
sales.select(col("customer_id"),col("product_id")).distinct()\
    .groupBy(col("customer_id"))\
    .agg(count("*").alias("cnt"))\
    .filter(col("cnt")>1).show(2)


+-----------+---+
|customer_id|cnt|
+-----------+---+
|         12|  2|
|          1|  7|
+-----------+---+
only showing top 2 rows



### 23. Find customers who purchased from multiple categories.

In [121]:
sales.alias("s")\
      .join(product.alias("p"),
            col("s.product_id")==col("p.product_id"),
            "inner"
      )\
      .groupBy("s.customer_id")\
      .agg(countDistinct(col("p.category")).alias("cnt"))\
      .filter(col("cnt")>1)\
      .show(3)

+-----------+---+
|customer_id|cnt|
+-----------+---+
|         12|  2|
|          1|  4|
|         13|  1|
+-----------+---+
only showing top 3 rows



### 24. Find customers who purchased from multiple brands.

In [126]:
sales.alias("s")\
      .join(product.alias("p"),
            (col("s.product_id")==col("p.product_id")),
            "inner"
      )\
      .groupBy(col("s.customer_id"))\
      .agg(countDistinct(col("p.brand")).alias("cnt"))\
      .filter(col("cnt")>1)\
      .show(3)


+-----------+---+
|customer_id|cnt|
+-----------+---+
|         12|  2|
|          1|  5|
|         13|  2|
+-----------+---+
only showing top 3 rows



### 25. Find customers who purchased in multiple months.

In [131]:
sales.groupBy(col("customer_id"))\
      .agg(countDistinct(month(col("sale_date"))).alias("cnt")  )\
      .filter(col("cnt")>1).show(3)

+-----------+---+
|customer_id|cnt|
+-----------+---+
|         12|  2|
|          1|  4|
|         13|  2|
+-----------+---+
only showing top 3 rows



### 26. Find repeat customers.

In [134]:
sales.groupBy(col("customer_id")).agg(count("*").alias("cnt")).filter(col("cnt")>1).show(3)

+-----------+---+
|customer_id|cnt|
+-----------+---+
|         12|  2|
|          1|  9|
|         13|  2|
+-----------+---+
only showing top 3 rows



### 27. Find one-time customers.

In [137]:
sales.groupBy(col("customer_id")).agg(count("*").alias("cnt")).filter(col("cnt")==1).show()

+-----------+---+
|customer_id|cnt|
+-----------+---+
+-----------+---+



### 28. Find average revenue per customer.

In [139]:
sales.groupBy(col("customer_id")).agg(avg(col("quantity")*col("unit_price")).alias("avg_revenue")).show(3)

+-----------+-----------------+
|customer_id|      avg_revenue|
+-----------+-----------------+
|         12|         130000.0|
|          1|72222.22222222222|
|         13|         105000.0|
+-----------+-----------------+
only showing top 3 rows



### 29. Find average revenue per product.

In [110]:
sales.show(1)

+-------+-----------+----------+--------+----------+--------+----------+
|sale_id|customer_id|product_id|store_id| sale_date|quantity|unit_price|
+-------+-----------+----------+--------+----------+--------+----------+
|      1|          1|       101|       1|2024-01-01|       1|     80000|
+-------+-----------+----------+--------+----------+--------+----------+
only showing top 1 row



In [109]:
customer.show(1)

+-----------+-------------+------+-----+------+----------+
|customer_id|customer_name|  city|state|gender| join_date|
+-----------+-------------+------+-----+------+----------+
|          1|         Amit|Mumbai|   MH|     M|2023-01-10|
+-----------+-------------+------+-----+------+----------+
only showing top 1 row



In [108]:
product.show(1)

+----------+------------+--------+-----+
|product_id|product_name|category|brand|
+----------+------------+--------+-----+
|       101|      iPhone|  Mobile|Apple|
+----------+------------+--------+-----+
only showing top 1 row



In [107]:
store.show(1)

+--------+--------------+------+------+
|store_id|    store_name|  city|region|
+--------+--------------+------+------+
|       1|Mumbai Central|Mumbai|  West|
+--------+--------------+------+------+
only showing top 1 row

